# 模型训练

运行完整的训练循环

In [ ]:
import sys; sys.path.append("..")
import os, torch
from torch.utils.data import DataLoader
from src.config import Config
from src.utils.seed import set_seed
cfg = Config(); set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from src.data.dataset import OralFluorescenceDataset
from src.data.transforms import get_train_transforms, get_val_transforms
tr_tf = get_train_transforms(); va_tf = get_val_transforms()
tr_ds = OralFluorescenceDataset(cfg.data.data_root, os.path.join(cfg.data.split_dir, cfg.data.train_split), transform=tr_tf, class_names=cfg.data.class_names)
va_ds = OralFluorescenceDataset(cfg.data.data_root, os.path.join(cfg.data.split_dir, cfg.data.val_split), transform=va_tf, class_names=cfg.data.class_names)
tr_l = DataLoader(tr_ds, cfg.training.batch_size, shuffle=True, num_workers=0)
va_l = DataLoader(va_ds, cfg.training.batch_size, shuffle=False, num_workers=0)
print(f"Train: {len(tr_ds)} | Val: {len(va_ds)}")

In [ ]:
from src.models.backbone import create_model
model = create_model(cfg)
p = sum(p.numel() for p in model.parameters())
print(f"Model: {cfg.model.backbone} | Params: {p:,}")

In [ ]:
from src.training.loss import get_loss_fn
from src.training.trainer import Trainer
criterion = get_loss_fn(cfg.training.loss)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.learning_rate, weight_decay=cfg.training.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.training.num_epochs)
trainer = Trainer(model, tr_l, va_l, criterion, optimizer, scheduler, cfg, device)
history = trainer.fit()

In [ ]:
# 绘制训练曲线
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train Loss"); ax1.plot(history["val_loss"], label="Val Loss"); ax1.legend(); ax1.grid(True)
ax2.plot(history["train_acc"], label="Train Acc"); ax2.plot(history["val_acc"], label="Val Acc"); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()
print(f"Best val acc: {trainer.best_val_acc:.2f}%")